In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# ============================
# CARREGAR TODOS OS CSVs
# ============================

results_path = Path("")

csv_files = list(results_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum CSV encontrado na pasta ../results")

dfs = []
for csv in csv_files:
    df_tmp = pd.read_csv(csv)
    df_tmp["source_file"] = csv.name  # opcional: rastrear origem
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# 🔧 garantir que erro é numérico
df["erro"] = pd.to_numeric(df["erro"], errors="coerce")

# ============================
# ORDENAR MODELOS
# ============================

order = (
    df.groupby("modelo")["erro"]
    .median()
    .sort_values()
    .index
    .tolist()
)

# ============================
# MAPA DE CORES
# ============================

color_map = {
    "baseline_ultra":  "#EF553B",
    "shape_ultra":  "#EF553B",
    "tail_ultra":  "#EF553B",
    "divergence_ultra": "#EF553B",
    "qderiv_ultra": "#EF553B",
    "baseline_tails_01": "#EF553B",
    "DyS_hellinger": "#fc03d3",
    "DyS_topsoe": "#fc03d3",
    "QuaDapt_DyS": "#fc03d3",
    "baseline_lite": "#19D3F3",
    "mfe": "#19D3F3",
    "qderiv_lite": "#19D3F3",
    "MiniRocket": "#19D3F3",
    "tsfresh": "#19D3F3",
    "catch22": "#19D3F3",
    "divergence_lite": "#19D3F3",
    "tail_lite": "#19D3F3",
    "shape_lite": "#19D3F3"
}

# ============================
# PLOT
# ============================

fig = px.box(
    df,
    x="modelo",
    y="erro",
    category_orders={"modelo": order},
    points="all",
    color="modelo",
    color_discrete_map=color_map
)

fig.update_traces(
    jitter=0.35,
    marker=dict(size=4, opacity=0.6),
)

fig.update_layout(
    title="Comparação de erro entre modelos (ordenado do melhor ao pior)",
    xaxis_title="Modelo",
    yaxis_title="Erro absoluto |prev_pred − prev_real|",
    template="simple_white",
    width=950,
    height=450,
    showlegend=True
)

fig

In [2]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# ============================
# CARREGAR TODOS OS CSVs
# ============================

results_path = Path("")

csv_files = list(results_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "Nenhum CSV encontrado"
    )

dfs = []

for csv in csv_files:

    df_tmp = pd.read_csv(csv)

    df_tmp["source_file"] = csv.name

    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# ============================
# LIMPEZA
# ============================

df["erro"] = pd.to_numeric(
    df["erro"],
    errors="coerce"
)

# ============================
# AGRUPAR TODOS OS MOSS
# ============================

df["grupo_modelo"] = df["modelo"]

df.loc[
    df["modelo"].str.contains(
        "MoSS",
        case=False,
        na=False
    ),
    "grupo_modelo"
] = "MoSS"

# ============================
# ORDEM DOS GRUPOS
# ============================

order = (
    df.groupby("grupo_modelo")["erro"]
    .median()
    .sort_values()
    .index
    .tolist()
)

# ============================
# CORES
# ============================

color_map = {
    "MoSS": "#19D3F3",
    "EMQ_BCTS": "#EF553B",
    "ClusterQuant": "#AB63FA"
}

# ============================
# PLOT
# ============================

fig = px.box(
    df,
    x="grupo_modelo",
    y="erro",
    color="grupo_modelo",
    category_orders={
        "grupo_modelo": order
    },
    color_discrete_map=color_map,
    points="all"
)

fig.update_traces(
    jitter=0.35,
    marker=dict(
        size=4,
        opacity=0.5
    )
)

fig.update_layout(
    title=(
        "Comparação entre grupos de modelos"
    ),
    xaxis_title="Grupo",
    yaxis_title=(
        "Erro absoluto |prev_pred − prev_real|"
    ),
    template="simple_white",
    width=900,
    height=500,
    showlegend=False
)

fig.show()

# ============================
# SALVAR HTML
# ============================

fig.write_html(
    "comparacao_grupos.html"
)

print(
    "✅ HTML salvo em comparacao_grupos.html"
)

✅ HTML salvo em comparacao_grupos.html


In [3]:
# ============================
# TABELA DE MEDIANAS
# ============================

tabela_mediana = (
    df.groupby(
        ["dataset", "grupo_modelo"]
    )["erro"]
    .median()
    .reset_index()
)

# pivotar para ficar bonito
tabela_mediana = tabela_mediana.pivot(
    index="dataset",
    columns="grupo_modelo",
    values="erro"
)

# ordenar datasets pela melhor mediana do MoSS
if "MoSS" in tabela_mediana.columns:
    tabela_mediana = tabela_mediana.sort_values(
        by="MoSS"
    )

print("\n===== MEDIANA POR DATASET =====\n")

print(
    tabela_mediana.round(4)
)

# salvar CSV
tabela_mediana.to_csv(
    "medianas_por_dataset.csv"
)

print(
    "\n✅ Tabela salva em medianas_por_dataset.csv"
)


===== MEDIANA POR DATASET =====

grupo_modelo          ClusterQuant  EMQ_BCTS    MoSS
dataset                                             
dry-bean.csv                0.0178    0.0088  0.0012
digits.csv                  0.0171    0.0119  0.0014
isolet.csv                  0.0136    0.0087  0.0017
image_seg.csv               0.0367    0.0082  0.0022
hand_digits.csv             0.0146    0.0059  0.0026
satellite.csv               0.0356    0.0173  0.0036
letter.csv                  0.0231    0.0060  0.0053
chess.csv                   0.0646    0.0206  0.0068
poker_hand.csv              0.1655    0.0799  0.0136
obesity.csv                 0.0524    0.0220  0.0213
abalone.csv                 0.0737    0.0800  0.0346
academic-success.csv        0.1658    0.0700  0.0353
wine-quality.csv            0.1580    0.0633  0.0377
molecular.csv               0.1139    0.0095  0.0384
waveform-v1.csv             0.0886    0.0347  0.0547
phishing.csv                0.1435    0.0235  0.0657
mhr.csv     

In [1]:
import pandas as pd
import plotly.express as px

# =========================================================
# LOAD
# =========================================================
results_path = Path("")

csv_files = list(results_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum CSV encontrado na pasta ../results")

dfs = []
for csv in csv_files:
    df_tmp = pd.read_csv(csv)
    df_tmp["source_file"] = csv.name  # opcional: rastrear origem
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# 🔧 garantir que erro é numérico
df["erro"] = pd.to_numeric(df["erro"], errors="coerce")

# =========================================================
# AGREGAÇÃO (modelo × dataset)
# 👉 troque mean por median se quiser
# =========================================================
df_agg = (
    df
    .groupby(["modelo", "dataset"], as_index=False)
    .agg(erro_mean=("erro", "mean"))
)

# =========================================================
# ORDEM DOS MODELOS (melhor → pior)
# =========================================================
order = (
    df_agg
    .groupby("modelo")["erro_mean"]
    .median()
    .sort_values()
    .index
    .tolist()
)


# =========================================================
# LINEPLOT
# =========================================================
fig = px.line(
    df_agg,
    x="dataset",
    y="erro_mean",
    color="modelo",
    category_orders={"modelo": order},
    markers=True
)

fig.update_layout(
    title="Erro médio por dataset (lineplot)",
    xaxis_title="Dataset",
    yaxis_title="Erro médio |prev_pred − prev_real|",
    template="simple_white",
    width=1100,
    height=500,
)

fig.update_traces(
    marker=dict(size=6),
    line=dict(width=2)
)

fig

NameError: name 'Path' is not defined

In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Carregar os dados
df = pd.read_csv("m_30_cluster_vs_emq_calibrated.csv")

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
# Criamos uma subfigura por dataset, em uma única coluna
fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02 # Espaço curto entre os gráficos
)

# 3. Iterar e adicionar cada gráfico com sua própria ordem
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    
    # Calcular a ordem local (pela mediana do erro neste dataset)
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()
    
    # Adicionar um boxplot para cada modelo, seguindo a ordem local
    for modelo in ordem_local:
        df_mod = df_ds[df_ds['modelo'] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod['erro'],
                name=modelo,
                boxpoints='outliers',
                legendgroup=modelo,
                showlegend=(i == 1) # Só mostra a legenda no primeiro gráfico
            ),
            row=i, col=1
        )

# 4. Ajustes finais de tamanho e estética
fig.update_layout(
    height=n_datasets * 400, # 300px para cada dataset
    template="plotly_white",
    title_text="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

# Deixar os eixos X independentes para cada subgráfico respeitar sua ordem
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="MAE")
fig.write_html("meu_resultado_ordenado.html")
fig.show()

In [2]:
mlquantify --version

NameError: name 'mlquantify' is not defined